In [4]:
import boto3
from braket.aws import AwsSession

# Setup environment

In [5]:
sess = AwsSession()
region = sess.region
role = sess.get_default_jobs_role()
default_bucket = sess.default_bucket()

prefix = "qml"

NOTE: If the step above fails with "RuntimeError: No default jobs roles found. Please ..." first create a default role in the AWS Console as described [here](https://docs.aws.amazon.com/braket/latest/developerguide/braket-jobs-first.html).

In [6]:
sts_client = boto3.client("sts")
account_id = sts_client.get_caller_identity()["Account"]

In [7]:
account_id

'456298966770'

# Build Cointainer

The quantum transfer learning approach used in this notebook is a hybrid classical-quantum algorithm, i.e. it uses alternatingly classical and quantum compute resources. Quantum compute resources are allocated through a job queue on Amazon Braket. Running a hybrid classical-quantum algorithm from the notebook instance would submit a quantum task to the queue on every classical-quantum iteration, and hence will lead to a potentially high walltime caused by accumulated queueing times. For avoiding high walltime the hybrid algorithm can be submitted as a [Amazon Braket hybrid job](https://aws.amazon.com/blogs/aws/introducing-amazon-braket-hybrid-jobs-set-up-monitor-and-efficiently-run-hybrid-quantum-classical-workloads/), which will give associated quantum tasks high priority once the job has started. 

In a Amazon Braket job the training code is executed in a container environment. Amazon Braket offers [prebuilt docker containers](https://github.com/aws/amazon-braket-containers) among others for QML with PyTorch which is used in this notebook. However, the pre-buit PyTorch container does not have the module torchvision installed, and hence, a custom container is used here. Setting up a custom container is also described in more detail [here](https://github.com/aws/amazon-braket-examples/tree/main/examples/hybrid_jobs/3_Bring_your_own_container).

A Dockerfile defines the environment the training script will run in. Here a prebuilt Amazon Braket PyTorch image is used as a baseline and the Python module torchvision is added to it.

In [8]:
%%writefile Dockerfile
FROM 292282985366.dkr.ecr.us-east-1.amazonaws.com/amazon-braket-pytorch-jobs:1.8.1-cpu-py37-ubuntu18.04

RUN python3 -m pip install --upgrade pip
RUN python3 -m pip install amazon-braket-sdk==1.35.5 --upgrade
RUN python3 -m pip install torchvision==0.10.1

Writing Dockerfile


To make the custom container available to Amazon Braket, an container image is created and stored in a repository of AWS Elastic Container Registry (ECR). The default Amazon Braket jobs have access to repositories starting with `amazon-braket`. For different names the poliy of the default role needs to be adjusted.

In [9]:
# create ECR
ecr_repository_name = "amazon-braket-my-qtc"
image_uri_byoc=f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repository_name}:latest"

In [10]:
!aws ecr create-repository --repository-name {ecr_repository_name} --region {region}


An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'amazon-braket-my-qtc' already exists in the registry with id '456298966770'


In [11]:
!docker login -u AWS -p $(aws ecr get-login-password --region us-east-1) 292282985366.dkr.ecr.us-east-1.amazonaws.com
!docker login -u AWS -p $(aws ecr get-login-password --region {region}) {account_id}.dkr.ecr.{region}.amazonaws.com

WARNING! Using --password via the CLI is insecure. Use --password-stdin.
WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded
WARNING! Using --password via the CLI is insecure. Use --password-stdin.
WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded


Let's build the custom Docker image defined by the Dockerfile and push it to the ECR repository. This makes the image ready to be used by Amazon Braket.

In [12]:
!docker build -t dockerfile .

[+] Building 0.0s (0/0)  docker:default
[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (2/3)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.1s
 => => transferring dockerfile: 295B                                       0.0s
 => [internal] load metadata for 292282985366.dkr.ecr.us-east-1.amazonaws  0.2s
 => [auth] sharing credentials for 292282985366.dkr.ecr.us-east-1.amazona  0.0s
[+] Building 0.4s (2/3)                                          docker:default
 => [internal] load build definition from Dockerfile                       0.1s
 => => transferring dockerfile: 295B                                       0.0s
 => [internal] load metadata for 292282985366.dkr.ecr.us-east-1.amazonaws  0.3s
 => [auth] sharing credentials for 292282985366.dkr.ecr.us-east-1.amazona  0.0s
[+] Building 0.4s (3/4)                                          docker:default


In [13]:
!docker tag dockerfile:latest {account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repository_name}:latest

In [14]:
!docker push {account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repository_name}:latest

The push refers to repository [456298966770.dkr.ecr.us-east-1.amazonaws.com/amazon-braket-my-qtc]

66b3eafa: Preparing 
5e3954b6: Preparing 
f7aef83f: Preparing 
f249220c: Preparing 
0d02a1e3: Preparing 
848677a4: Preparing 
0c379989: Preparing 
7b217894: Preparing 
5d7250cc: Preparing 
809cab7c: Preparing 
44db4ae6: Preparing 
54dc9d6a: Preparing 
1f6db62f: Preparing 
ed811fbf: Preparing 
d0ec182e: Preparing 
75ff1277: Preparing 
2217b2a2: Preparing 
a5bb4263: Preparing 
53fcb889: Preparing 
6b3eafa: Pushed   2.712GB/2.708GB8APushing  1.621GB/2.708GBPushing  2.693GB/2.708GBlatest: digest: sha256:42c7c3d6be08c0e93a1d8a6d83728c234b2abc0faf7aca07190e2f1163d415ac size: 4532
